Embedding 嵌入：从 `nn.Embedding` 到语义向量与向量数据库

Embedding（嵌入）是把离散对象映射到连续向量空间的表示方法。词、子词、用户 ID、商品 ID、类别编号、图节点都可以映射为稠密向量，让神经网络通过向量运算学习对象之间的关系。

本文按从基础到进阶的顺序学习：离散 ID 与 one-hot、Embedding 查表原理、`nn.Embedding`、梯度与稀疏更新、文本/推荐/类别 embedding、预训练向量、向量相似度与近邻检索、FAISS/向量数据库，以及面向 RAG 的检索流程。先要记住一个重要区分：

```text
nn.Embedding：模型中的可训练参数表，输入整数 ID，查出稠密向量。
向量数据库：保存文档/对象向量并建立近邻索引，用于相似度检索和元数据过滤。
```

二者都处理向量，但不是同一种组件：Embedding 层负责“表示如何学习/查询”，向量数据库负责“如何存储和检索大量已经生成的向量”。

示例默认适用于 PyTorch 2.x；向量数据库部分会区分纯 NumPy 原理示例、FAISS 与 Chroma 等可选依赖。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"device: {device}")

In [ ]:
## 4. Embedding 的核心思想

# Embedding 为每个离散 ID 学习一个 `D` 维稠密向量：
# - `V`：词表/实体总数。
# - `D`：embedding dimension，每个对象的向量维度。
# - `E`：embedding table，整个查找矩阵。
# - `E_i`：ID `i` 对应的向量。
#
# 例如 5 个 token、每个用 3 个浮点数表示，Embedding 表就是 `[5,3]`：
embed = torch.randn(5, 3)
print(embed.shape)

# 嵌入的权重矩阵大小是 [N, D]，N表示词汇表大小，D表示嵌入维度
embedding_table = nn.Embedding(num_embeddings=5, embedding_dim=3)
print(f"embedding_table.weight.shape : {embedding_table.weight.shape}")
token_ids = torch.tensor([1,2,3])
vectors = embedding_table(token_ids)
print(vectors, vectors.shape)

In [ ]:
## 5. Embedding 是查表，不是神秘的神经网络
# 相同 token_id 返回的嵌入向量相同
table = nn.Embedding(6, 2)
ids = torch.tensor([4, 1, 4])
vectors = table(ids)

print(vectors, vectors.shape)
# torch.allclose() 是 PyTorch 中用于判断两个 Tensor 是否近似相等的函数。
print(torch.allclose(vectors[0], vectors[2]))  # 同一 ID 查到同一行
print(torch.allclose(vectors[0], table.weight[4]))  # 与权重表第 4 行相同


# 查表不含对 ID 的连续插值，也不会因为 ID 从 4 变为 5 就保证向量只改变一点。ID 需要是合法的整数索引：
print(table(torch.tensor([4])))
print(table(torch.tensor([5])))
# print(table(torch.tensor([6]))) # IndexError: index out of range in self

# 假设输入语句是批量的
ids = torch.tensor([
    [1, 2, 0],
    [3, 4, 1]
], dtype=torch.long)
embedding = nn.Embedding(num_embeddings=5, embedding_dim=8)
out = embedding(ids)
print(out.shape)  # [B=2, L=3, D=8]，批量是2，序列长度是3，每一个token的嵌入维度是8

# 输入 dtype 通常用 `torch.long` / `torch.int64`；ID 最小为 0，最大必须小于 `num_embeddings`。

In [ ]:
## 6. `nn.Embedding` 最小例子
# `num_embeddings` 是可能的 ID 数量，不是 batch size，也不是序列长度。`embedding_dim` 是每个 ID 输出的特征维数。
embedding = nn.Embedding(num_embeddings=5, embedding_dim=8)



In [ ]:
## 7. 输入 shape 与输出 shape
embedding_table = nn.Embedding(num_embeddings=10, embedding_dim=8)

# 输入shape [seq_len] ，输出shape [seq_len, embedding_dim]
# 如输入是 [3]，输出则是 [3, 8]
print(embedding_table(torch.tensor([1, 2, 3], dtype=torch.long)).shape)
# 输入 shape [batch_size, seq_len] 输出 shape [batch_size, seq_len, embedding_dim]
# 如输入[2, 3], 输出 [2, 3, 8]
print(embedding_table(torch.tensor([[1,2,3], [5,6,7]], dtype=torch.long)).shape)

# 由以上可知，输出会在输入的基础上增加最里面一维，为嵌入维度
# input:  [*]        # 任意数量的 ID 维度
# output: [*, D]     # 原 shape 后追加 embedding_dim

In [ ]:
## 8. `nn.Embedding` 权重是什么 vocabulary
# Embedding 的权重是形状 `[V,D]` 的可训练参数：，V表示词汇表大小，D表示嵌入维度

embedding = nn.Embedding(10, 4)
print(embedding.weight.shape)
print(embedding.weight.requires_grad)  # True
print(sum(p.numel() for p in embedding.parameters()))  # 40
# 默认权重通常从随机分布初始化。训练时，loss 对被使用的行产生梯度，optimizer 更新这些向量。

ids = torch.tensor([2, 4, 2])
vectors = embedding(ids)
loss = vectors.square().mean()
print(embedding.weight.grad)  # [10, 4]
loss.backward()
print(embedding.weight.grad.shape)  # [10, 4] ，经过 loss.backward() 之后 embedding.weight 里面的参数已经被求导了



In [ ]:
## 9. Embedding 与 Linear 层的等价关系
# 当 one-hot 作为 Linear 输入且 Linear 无 bias 时，输出等于对应 embedding 行。`nn.Embedding` 避免构造大型 one-hot Tensor：
vocab_size, dim = 7, 5
linear = nn.Linear(in_features=vocab_size, out_features=dim, bias=False)
embedding = nn.Embedding(vocab_size, dim)

with torch.no_grad():
    embedding.weight.copy_(linear.weight.T)

ids = torch.tensor([1, 3, 6])
one_hot = F.one_hot(ids, num_classes=vocab_size).float()
print(one_hot, one_hot.shape)
print(linear(one_hot), linear(one_hot).shape)
print(embedding(ids), embedding(ids).shape)
print(torch.allclose(linear(one_hot), embedding(ids)))  # True

In [ ]:
## 11. `padding_idx`：让 PAD 向量保持固定
# 文本 batch 常把短句补齐到统一长度。可设置 `padding_idx`，使该行不参与梯度更新，并在默认初始化下保持零向量：
embedding = nn.Embedding(num_embeddings=10, embedding_dim=4, padding_idx=0)
src_ids = [
    [1, 2, 0, 1, 3],
    [3, 4, 1, 5],
    [5, 6, 7]
]
# batch 中每一个seq的长度必须一致才能通过embedding
# ValueError: expected sequence of length 5 at dim 1 (got 4)
# print(embedding(src_ids))

# 因此，如果是batch传入embedding的时候，必须在传入之前做pad

src_ids = [
    [1, 2, 0, 1, 3],
    [3, 4, 1, 5],
    [5, 6, 7]
]
def padding(src_ids, padding_idx = 0):
    if src_ids is None or len(src_ids) == 0:
        return torch.tensor([])

    # 找最大的一行
    max_len = len(src_ids[0])
    for ids in src_ids:
        cur_len = len(ids)
        if cur_len > max_len:
            max_len = cur_len
    # max_len = max(len(ids) for ids in src_ids)

    # 填充
    res_ids = []
    for ids in src_ids:
        item = []
        for j in ids:
            item.append(j)
        for j in range(max_len - len(ids)):
            item.append(padding_idx)
        res_ids.append(item)
    return torch.tensor(res_ids)

paded_ids = padding(src_ids, 0)
print(paded_ids, paded_ids.shape)

emd = embedding(paded_ids)
print(emd.shape)
